# 02 Train M8 (heavy)

Trains one M8 bundle per fold, eighteen in all. This is the only heavy stage of the evaluation and the one the author runs by hand. It is resumable: a fold whose bundle already exists is skipped.

Abbreviations used here: **RPF** is reverse power flow, the condition where a distribution substation exports power because rooftop solar exceeds local demand; a *wrong RPF sign* is a meter recording that stores the export as an import. **M7** is the deterministic threshold rule, **M8** the two-stage XGBoost classifier and **M9** the compact counterfactual method (revision 2). **MW** and **MWh** are megawatts and megawatt-hours; one interval is 15 minutes.

**Inputs.** The fold manifest from notebook 01 and the two frozen datasets.

**Outputs.** `outputs/01_final_evaluation/02_bundles/<fold_id>/...` (the pickled bundles, gitignored) and `02_bundles/<fold_id>.json` (kept: training stations, row counts, in-bundle validation metrics, bundle path); `02_bundles/training_summary.csv`; the manifest `manifests/02_train_m8.json`.

**Approximate runtime.** About two minutes per fold on a laptop, so roughly forty minutes for all eighteen.

**Prerequisites.** Notebook 01.

**Main process.**

1. Load the configuration and the folds.
2. For each fold, assemble the training frame (every other station, both cohorts, Beta `sure` days only, complete days), assert the held-out station is absent, and train through `pynrpf.api.train_m8_xgb` with the frozen thresholds.
3. Record the bundle manifest per fold and the training summary.

## 1. Setup

In [ ]:
import sys
from pathlib import Path

import pandas as pd
from IPython.display import Image, Markdown, display


def article_root() -> Path:
    """Locate publication/2_journal_article from JupyterLab, VS Code or the repository root."""
    for candidate in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
        if (candidate / "final_eval" / "cli.py").exists():
            return candidate
        nested = candidate / "publication" / "2_journal_article"
        if (nested / "final_eval" / "cli.py").exists():
            return nested
    raise FileNotFoundError("Could not locate publication/2_journal_article.")


ARTICLE = article_root()
sys.path.insert(0, str(ARTICLE))
sys.path.insert(0, str(ARTICLE.parents[1] / "src"))  # the repository's pynrpf package

from final_eval import cli, config  # noqa: E402

SETTINGS = config.load()  # verifies the frozen dataset hashes
OUT = SETTINGS.output_root()
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)
print("Article root:", ARTICLE.relative_to(ARTICLE.parents[2]))

## 2. What M8 is fitted on

The fit window is 2021-11-01 to 2024-05-31 and the in-bundle validation window 2024-06-01 to 2024-09-30, both on other-station rows only; nothing is tuned on the validation window, it only reports the bundle's own metrics. The thresholds 0.586 (day) and 0.892 (interval) are frozen from the conference configuration. Change nothing here: the settings are the configuration file.

In [ ]:
display(pd.Series(SETTINGS["m8"]["split"], name="split"))
display(pd.Series(SETTINGS["m8"]["thresholds"], name="thresholds"))

## 3. Train

Set `ONLY_FOLD` to one fold id (for example `"beta_beta_A"`) to train a single fold, or leave it `None` for all eighteen. Set `FORCE = True` to retrain folds whose bundle exists. Progress is printed per fold.

In [ ]:
ONLY_FOLD = None  # e.g. "beta_beta_A"
FORCE = False

summary = cli.stage_train_m8(SETTINGS, fold_id=ONLY_FOLD, force=FORCE)
display(summary[["fold_id", "held_out", "n_training_rows", "n_training_rpf_days", "elapsed_s", "skipped"]])

## 4. In-bundle validation metrics

These are the package's own metrics on the validation window of the *training* stations. They say whether a fit is sane; they are not held-out results and are not reported in the paper.

In [ ]:
import json

rows = []
for path in sorted((OUT / "02_bundles").glob("*.json")):
    record = json.loads(path.read_text(encoding="utf-8"))
    vm = record.get("validation_metrics", {})
    rows.append({"fold_id": record["fold_id"],
                 **{f"day_{k}": v for k, v in vm.get("xgb1_day", {}).items() if k in ("precision", "recall", "f1")},
                 **{f"interval_{k}": v for k, v in vm.get("xgb2_timestamp", {}).items() if k in ("precision", "recall", "f1")}})
display(pd.DataFrame(rows).round(3))

## Conclusion

With all eighteen bundle manifests present, notebook 04 can score every held-out station. If a fold failed, re-run this notebook: completed folds are skipped.